### Functions

In [1]:
def load_session_data(subject, date):
    """Load all data for a given subject and date"""
    import sys
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig
    
    # Load session
    loader = NeuralDataLoader()
    loader.load_session(subject, date)
    config = Dots3DMPConfig(subject)

    # spike data (unit, trial, time)
    stimOn_spikes = loader.get_spike_data(alignment='stimOn', good_units_only=True, good_trials_only=True)
    saccOnset_spikes = loader.get_spike_data(alignment='saccOnset', good_units_only=True, good_trials_only=True)
    postTargHold_spikes = loader.get_spike_data(alignment='postTargHold', good_units_only=True, good_trials_only=True)
    tuning_spikes = loader.get_tuning_data(good_units_only=True, good_trials_only=True)

    # behavioral data
    behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True)
    behavior_tuning = loader.get_behavioral_data(task='tuning', good_trials_only=True)
    behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
    behavior_tuning_converted = config.convert_behavioral_data(behavior_tuning, task='tuning')

    # Unit Info
    unit_info = loader.get_unit_info(good_units_only=True)
    MST_units = loader.get_units_by_area(unit_info, area_name='MST')
    VPS_units = loader.get_units_by_area(unit_info, area_name='VPS')
    dual_units = loader.get_units_by_area(unit_info, area_name='dual')

    # Time Info
    time_info = config.get_time_Info('dots3DMP')
    time_info_tuning = config.get_time_Info('tuning')
    time_axes_dots3DMP = config.get_time_axes('dots3DMP')

    # Prepare data
    spikes_data = {
        'stimOn': stimOn_spikes,
        'saccOnset': saccOnset_spikes,
        'postTargHold': postTargHold_spikes
    }
    
    units_data = {
        'MST': MST_units,
        'VPS': VPS_units,
        'dual': dual_units
    }
    
    
    return {
        'loader': loader,
        'config': config,
        'spikes_data': spikes_data,
        'behavior_converted': behavior_converted,
        'behavior_tuning_converted': behavior_tuning_converted,
        'unit_info': unit_info,
        'units_data': units_data,
        'time_axes_dots3DMP': time_axes_dots3DMP,
        'time_info': time_info,
        'time_info_tuning': time_info_tuning
    }

In [2]:
def run_sliding_window_decoding(subject, date, data_dict, permutation_test=False, partial_regress=False):
    """Run sliding window decoding analysis"""
    from analysis_pseudopopulation import SlidingWindowDecoder
    import gc
    from IPython.display import clear_output
    
    areas = ['MST', 'VPS']
    decode_targets = ['stimulus','choice', 'PDW']
    
    # Run decoding analysis
    for area in areas:
        print(f"\n=== Processing area: {area} ===")
        
        valid_units = data_dict['units_data'][area]
        
        for target in decode_targets:
            print(f"\n--- Processing target: {target} ---")
            
            # Create decoder instance for this specific area/target combination
            decoder = SlidingWindowDecoder(subject, date, run_permutation_test=permutation_test, 
                                          partial_regress=partial_regress, n_permutations=20)
            
            try:
                for mod in [1, 2, 3]:
                    for coh in [1, 2]:
                        if mod == 1 and coh == 2:
                            continue  

                        results = decoder.run_decoding_analysis_cv(
                            spikes_data=data_dict['spikes_data'],
                            behavior_data=data_dict['behavior_converted'],
                            time_axes=data_dict['time_axes_dots3DMP'],
                            area=area,
                            target=target, 
                            train_mod=mod, 
                            train_coh=coh,    
                            test_mod=mod, 
                            test_coh=coh,
                            valid_units=valid_units,
                            save_results=True
                        )
                        
                        # Clear results immediately after they're saved
                        if results:
                            results.clear()
                            del results
                        
                        gc.collect()  # Clean up after each mod/coh combination
                        
                        # Clear output to prevent memory buildup
                        clear_output(wait=True)
                        print(f"Completed: {subject} {date} | {area} | {target} | mod{mod} coh{coh}")
            
            finally:
                # Clean up decoder instance after each target
                del decoder
                gc.collect()
    
    print(f"✓ Completed sliding window decoding for {subject} {date}")

In [3]:
def process_all_dates(subject, dates, run_sliding_window=True, permutation_test=False, partial_regress=False):
    """Process all dates with optional sliding window decoding"""
    
    for date in dates:
        print(f"\n{'#'*80}")
        print(f"PROCESSING DATE: {date}")
        print(f"{'#'*80}")
        
        try:
            # Step 1: Load data
            print("Loading session data...")
            data_dict = load_session_data(subject, date)

            # Step 3: Run sliding window decoding (optional)
            if run_sliding_window:
                print("Running sliding window decoding...")
                run_sliding_window_decoding(subject, date, data_dict, permutation_test=permutation_test, partial_regress=partial_regress)
                
        except Exception as e:
            print(f"Error processing date {date}: {e}")
            continue
    
    return

### Main code, run here

In [4]:
subject = "zarya"
dates = ["20250602", "20250702", "20250710", "20250523", "20250501", "20250417"]


# Run sliding window decoding
process_all_dates(subject, dates, run_sliding_window=True, permutation_test=True, partial_regress=True)
# process_all_dates(subject, dates, run_sliding_window=True, permutation_test=True, partial_regress=False)

Completed: zarya 20250417 | VPS | PDW | mod3 coh2
✓ Completed sliding window decoding for zarya 20250417
